# Online Product Recommendation System Using SVD

## ระบบแนะนำสินค้าออนไลน์สไตล์ Shopee/Lazada

### วัตถุประสงค์ของโครงงาน

โครงงานนี้มีเป้าหมายเพื่อสร้างระบบแนะนำสินค้าออนไลน์อย่างง่าย โดยใช้แนวคิดจากพีชคณิตเชิงเส้น (Linear Algebra) และเทคนิค **Singular Value Decomposition (SVD)** เพื่อค้นหา **latent features** หรือคุณลักษณะแฝงจากพฤติกรรมของลูกค้า

ข้อมูลที่ใช้เป็น **implicit feedback** หมายถึงข้อมูลพฤติกรรมที่ไม่ได้ให้คะแนนโดยตรง แต่สะท้อนความสนใจของผู้ใช้ เช่น การดูสินค้า การเพิ่มลงตะกร้า หรือการซื้อสินค้า โดยกำหนดค่าดังนี้

- `0` = ไม่เคยดูหรือไม่มีปฏิสัมพันธ์

- `1` = ดูสินค้า

- `2` = กดถูกใจ

- `3` = เพิ่มลงตะกร้า

- `4` = รอโอนเงิน

- `5` = ซื้อสินค้า

แนวคิดสำคัญคือ ถ้าเรามองผู้ใช้และสินค้าเป็นเมทริกซ์ เราสามารถใช้ SVD แยกโครงสร้างหลักของข้อมูลออกมา แล้วนำไปประมาณค่าความสนใจในสินค้าที่ผู้ใช้ยังไม่เคยมีปฏิสัมพันธ์ได้

## Step 1: Data Preparation / Matrix Representation

ในระบบแนะนำสินค้า เราสามารถแทนข้อมูลจริงให้อยู่ในรูปของ **เมทริกซ์ (Matrix)** ได้ โดยให้

- แถว (rows) แทนผู้ใช้หรือลูกค้า

- หลัก (columns) แทนสินค้า

- ค่าในเมทริกซ์แทนระดับปฏิสัมพันธ์ระหว่างผู้ใช้กับสินค้า

เมทริกซ์นี้เรียกว่า **User-Product Matrix** หรือเมทริกซ์ $R$ ซึ่งเป็นการแปลงพฤติกรรมจริงในโลกออนไลน์ให้อยู่ในรูปที่สามารถคำนวณทางคณิตศาสตร์ได้

In [2]:
import numpy as np

products = ["เสื้อยืด", "หูฟังไร้สาย", "รองเท้าผ้าใบ", "กระเป๋าเป้", "คีย์บอร์ดเกมมิ่ง"]

R = np.array([
    [5, 0, 4, 0, 1],
    [4, 0, 5, 0, 0],
    [0, 5, 0, 4, 0],
    [0, 4, 0, 5, 1]
], dtype=float)

print("สินค้า:", products)
print(R)

สินค้า: ['เสื้อยืด', 'หูฟังไร้สาย', 'รองเท้าผ้าใบ', 'กระเป๋าเป้', 'คีย์บอร์ดเกมมิ่ง']
[[5. 0. 4. 0. 1.]
 [4. 0. 5. 0. 0.]
 [0. 5. 0. 4. 0.]
 [0. 4. 0. 5. 1.]]


จากเมทริกซ์ $R$ จะเห็นว่า ผู้ใช้แต่ละคนมีรูปแบบความสนใจที่แตกต่างกัน เช่น ผู้ใช้บางคนสนใจสินค้ากลุ่มแฟชั่น ขณะที่อีกกลุ่มสนใจสินค้าไอที การแทนข้อมูลเป็นเมทริกซ์ทำให้เราสามารถใช้เครื่องมือของ Linear Algebra เช่น การแยกเมทริกซ์ การคูณเมทริกซ์ และการวัดความคล้ายกันของเวกเตอร์ เพื่อวิเคราะห์ความสัมพันธ์เหล่านี้ได้

## Step 2: SVD Computation

- `U` คือ left singular matrix หรือเมทริกซ์ฝั่งผู้ใช้
- `S` คือ array ของ singular values ที่เรียงจากมากไปน้อย
- `Vt` คือ right singular matrix แบบ transpose หรือ $V^T$

ในเชิง Linear Algebra เมทริกซ์ `U` และ `Vt` มีสมบัติเป็น orthogonal matrix กล่าวคือ เมื่อนำ `U @ U.transpose()` จะได้เมทริกซ์เอกลักษณ์โดยประมาณ นอกจากนี้เราสามารถตรวจสอบ singular values ได้ด้วยแนวคิดจากไฟล์เรียน คือคำนวณ

$$U^T R V$$

ซึ่งในโค้ดเขียนเป็น `U.transpose() @ R @ Vt.transpose()` ผลลัพธ์ควรเป็นเมทริกซ์แนวทแยงที่มีค่า singular values อยู่บนเส้นทแยงมุม

In [3]:
np.set_printoptions(precision=3, suppress=True)

U, S, Vt = np.linalg.svd(R, full_matrices=False)

print("Matrix U:")
print(U)

print("=" * 30)
print("Singular Values (S):")
print(S)

print("=" * 30)
print("Matrix Vt หรือ V^T:")
print(Vt)

print("=" * 30)
print("ตรวจสอบว่า U เป็น orthogonal matrix: U @ U.transpose()")
print(U @ U.transpose())

print("=" * 30)
print("ตรวจสอบ singular values จาก U.T @ R @ Vt.T")
BB = U.transpose() @ R @ Vt.transpose()
print(BB)

Matrix U:
[[ 0.506  0.5   -0.494 -0.5  ]
 [ 0.494  0.5    0.506  0.5  ]
 [ 0.494 -0.5    0.506 -0.5  ]
 [ 0.506 -0.5   -0.494  0.5  ]]
Singular Values (S):
[9.056 9.    1.41  1.   ]
Matrix Vt หรือ V^T:
[[ 0.498  0.496  0.496  0.498  0.112]
 [ 0.5   -0.5    0.5   -0.5    0.   ]
 [-0.315  0.395  0.395 -0.315 -0.7  ]
 [-0.5   -0.5    0.5    0.5    0.   ]]
ตรวจสอบว่า U เป็น orthogonal matrix: U @ U.transpose()
[[ 1.  0.  0. -0.]
 [ 0.  1. -0.  0.]
 [ 0. -0.  1. -0.]
 [-0.  0. -0.  1.]]
ตรวจสอบ singular values จาก U.T @ R @ Vt.T
[[ 9.056  0.    -0.     0.   ]
 [ 0.     9.     0.    -0.   ]
 [ 0.    -0.     1.41  -0.   ]
 [-0.    -0.    -0.     1.   ]]


## Step 3: Dimensionality Reduction & Reconstruction

หลังจากได้ `U`, `S`, และ `Vt` แล้ว เราจะทำ **Truncated SVD** ตามแนวคิดเดียวกับตัวอย่าง image compression ในไฟล์เรียน คือเลือกใช้เฉพาะ singular values จำนวน `k` ค่าแรก

ในโปรเจกต์นี้เลือก `k = 2` เพราะข้อมูลตัวอย่างมีแนวโน้มความสนใจหลักประมาณ 2 กลุ่ม ได้แก่ กลุ่ม Fashion และกลุ่ม IT/Gadget การเก็บ 2 มิติแรกจึงช่วยรักษาโครงสร้างหลักของข้อมูลไว้ พร้อมลด noise หรือรายละเอียดที่ไม่จำเป็น

สูตรที่ใช้คือ

$$\hat{R} = U_k \Sigma_k V_k^T$$

ในโค้ดจะเขียนให้เหมือนตัวอย่างไฟล์เรียนมากขึ้นเป็น

```python
R_hat = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
```

โดย `R_hat` คือเมทริกซ์คะแนนที่สร้างกลับขึ้นมาใหม่จาก latent features จำนวน `k` มิติ และใช้เป็นคะแนนทำนายสำหรับการแนะนำสินค้า

In [4]:
k = 2
R_hat = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
R_hatNok = U @ np.diag(S) @ Vt

print("R:")
print(R)

print("=" * 30)

print("R ที่ไม่ได้ลดมิติ:")
print(np.round(R_hatNok, 3))

print("=" * 30)


print("R หลังจากการลดมิติด้วย SVD (ใช้ k=2):")
print(np.round(R_hat, 3))

R:
[[5. 0. 4. 0. 1.]
 [4. 0. 5. 0. 0.]
 [0. 5. 0. 4. 0.]
 [0. 4. 0. 5. 1.]]
R ที่ไม่ได้ลดมิติ:
[[ 5.  0.  4.  0.  1.]
 [ 4.  0.  5.  0.  0.]
 [ 0.  5.  0.  4. -0.]
 [-0.  4.  0.  5.  1.]]
R หลังจากการลดมิติด้วย SVD (ใช้ k=2):
[[ 4.531  0.025  4.525  0.031  0.512]
 [ 4.475 -0.032  4.468 -0.025  0.5  ]
 [-0.025  4.468 -0.032  4.475  0.5  ]
 [ 0.031  4.525  0.025  4.531  0.512]]


ค่าที่ได้ใน $\hat{R}$ เป็นคะแนนที่ระบบประมาณขึ้นจาก latent features แม้ว่าบางช่องใน $R$ เดิมจะเป็น 0 แต่ใน $\hat{R}$ อาจมีค่ามากกว่า 0 เพราะระบบคาดการณ์ว่าผู้ใช้อาจสนใจสินค้านั้นจากรูปแบบความสัมพันธ์ของผู้ใช้และสินค้าอื่น ๆ

## Step 4: Product Recommendation

ในขั้นตอนนี้เราจะเลือก **User index 1** หรือ **Customer 2** แล้วแนะนำสินค้าที่ลูกค้ายังไม่เคยมีปฏิสัมพันธ์ โดยใช้คะแนนที่ทำนายจาก $\hat{R}$

หลักการคือ

1. ดูแถวของ Customer 2 ในเมทริกซ์เดิม $R$

2. ตัดสินค้าที่มีคะแนนมากกว่า 0 ออก เพราะเป็นสินค้าที่เคยซื้อหรือเคยมีปฏิสัมพันธ์แล้ว

3. เลือกสินค้าที่ยังไม่เคยมีปฏิสัมพันธ์และมี predicted score สูงที่สุดใน $\hat{R}$

In [5]:
user_index = int(input("กรุณาเลือก Customer (1-4)เพื่อดูสินค้าที่แนะนำ: ")) - 1

In [6]:
original_scores = R[user_index]
predicted_scores = R_hat[user_index]


# กรองสินค้าที่เคยมีปฏิสัมพันธ์แล้วออก
candidate_scores = predicted_scores.copy()
candidate_scores[original_scores > 0] = -np.inf

# เลือกสินค้าที่มีคะแนนทำนายสูงที่สุด
recommended_index = np.argmax(candidate_scores)
recommended_product = products[recommended_index]
recommended_score = predicted_scores[recommended_index]

print("Customer 2 original interaction:")
for product, score in zip(products, original_scores):
    print("- {}: {:.0f}".format(product, score))

print("="*30)

print("Predicted scores from R_hat:")
for product, score in zip(products, predicted_scores):
    status = "เคยมีปฏิสัมพันธ์แล้ว" if original_scores[products.index(product)] > 0 else "ยังไม่เคยมีปฏิสัมพันธ์"
    print("- {}: {:.3f} ({})".format(product, score, status))
    
print("="*30)

print("ผลลัพธ์การแนะนำสินค้า")
print("แนะนำให้ Customer 2: {}".format(recommended_product))
print("Predicted score: {:.3f}".format(recommended_score))

Customer 2 original interaction:
- เสื้อยืด: 5
- หูฟังไร้สาย: 0
- รองเท้าผ้าใบ: 4
- กระเป๋าเป้: 0
- คีย์บอร์ดเกมมิ่ง: 1
Predicted scores from R_hat:
- เสื้อยืด: 4.531 (เคยมีปฏิสัมพันธ์แล้ว)
- หูฟังไร้สาย: 0.025 (ยังไม่เคยมีปฏิสัมพันธ์)
- รองเท้าผ้าใบ: 4.525 (เคยมีปฏิสัมพันธ์แล้ว)
- กระเป๋าเป้: 0.031 (ยังไม่เคยมีปฏิสัมพันธ์)
- คีย์บอร์ดเกมมิ่ง: 0.512 (เคยมีปฏิสัมพันธ์แล้ว)
ผลลัพธ์การแนะนำสินค้า
แนะนำให้ Customer 2: กระเป๋าเป้
Predicted score: 0.031


## Step 5: User Similarity using Cosine Similarity

นอกจาก SVD แล้ว เราสามารถวัดความคล้ายกันของผู้ใช้ได้ด้วย **Cosine Similarity** ซึ่งเป็นการวัดมุมระหว่างเวกเตอร์สองตัว

ถ้าให้เวกเตอร์ผู้ใช้สองคนเป็น $u$ และ $v$ สูตรคือ

$$\text{cosine similarity}(u, v) = \frac{\langle u, v \rangle}{\|u\|\|v\|}$$

โดยที่ $\langle u, v \rangle$ คือ **inner product** หรือ dot product และ $\|u\|$ คือความยาวของเวกเตอร์

แนวคิดนี้เชื่อมกับ **Inner Product Space** เพราะ dot product ใช้วัดทั้งขนาดและทิศทางของเวกเตอร์ ถ้า cosine similarity มีค่าใกล้ 1 แปลว่าเวกเตอร์มีทิศทางใกล้เคียงกัน หรือผู้ใช้มีพฤติกรรมคล้ายกันมาก

In [7]:
def cosine_similarity(u, v):
    """คำนวณ Cosine Similarity ระหว่างเวกเตอร์ u และ v"""
    dot_product = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    
    if norm_u == 0 or norm_v == 0:
        return 0
    return dot_product / (norm_u * norm_v)
user_1 = R[0]
user_2 = R[1]
user_3 = R[2]
user_4 = R[3]
sim_user1_user2 = cosine_similarity(user_1, user_2)
sim_user1_user3 = cosine_similarity(user_1, user_3)
sim_user1_user4 = cosine_similarity(user_1, user_4)
sim_user2_user3 = cosine_similarity(user_2, user_3)
sim_user2_user4 = cosine_similarity(user_2, user_4)
sim_user3_user4 = cosine_similarity(user_3, user_4)

print("Cosine Similarity ระหว่าง User 1 และ User 2: {:.3f}".format(sim_user1_user2))
print("Cosine Similarity ระหว่าง User 1 และ User 3: {:.3f}".format(sim_user1_user3))
print("Cosine Similarity ระหว่าง User 1 และ User 4: {:.3f}".format(sim_user1_user4))
print("Cosine Similarity ระหว่าง User 2 และ User 3: {:.3f}".format(sim_user2_user3))
print("Cosine Similarity ระหว่าง User 2 และ User 4: {:.3f}".format(sim_user2_user4))
print("Cosine Similarity ระหว่าง User 3 และ User 4: {:.3f}".format(sim_user3_user4))

Cosine Similarity ระหว่าง User 1 และ User 2: 0.964
Cosine Similarity ระหว่าง User 1 และ User 3: 0.000
Cosine Similarity ระหว่าง User 1 และ User 4: 0.024
Cosine Similarity ระหว่าง User 2 และ User 3: 0.000
Cosine Similarity ระหว่าง User 2 และ User 4: 0.000
Cosine Similarity ระหว่าง User 3 และ User 4: 0.964


จากผลลัพธ์ที่ได้ User 1 และ User 2 ควรมีค่า cosine similarity สูง เพราะทั้งสองคนมีปฏิสัมพันธ์กับสินค้าแฟชั่นคล้ายกัน เช่น เสื้อยืดและรองเท้าผ้าใบ ในขณะที่ User 1 และ User 3 มีความคล้ายกันต่ำกว่า เพราะ User 3 สนใจสินค้าคนละกลุ่มมากกว่า

## Step 6: Category Preference using Vector Weighted Sum

ในขั้นตอนสุดท้าย เราจะวิเคราะห์ความชอบหมวดหมู่สินค้าโดยใช้เวกเตอร์ฐานแบบง่าย ๆ สำหรับหมวดสินค้า

- Fashion = `[1, 0, 1, 0, 0]` หมายถึง เสื้อยืดและรองเท้าผ้าใบ

- IT = `[0, 1, 0, 0, 1]` หมายถึง หูฟังไร้สายและคีย์บอร์ดเกมมิ่ง

เมื่อนำเวกเตอร์ของ User 1 มา dot product กับเวกเตอร์หมวดหมู่ จะได้คะแนนรวมแบบถ่วงน้ำหนักของความสนใจในหมวดนั้น

มุมมองทาง Linear Algebra คือ เวกเตอร์ความสนใจของผู้ใช้สามารถมองเป็น **Linear Combination** ของเวกเตอร์หมวดหมู่ต่าง ๆ ได้ เช่น ผู้ใช้หนึ่งคนอาจมีองค์ประกอบของความสนใจด้าน Fashion และ IT ผสมกันในสัดส่วนที่แตกต่างกัน

In [8]:
user_vector = int(input("กรุณาเลือก Customer (1-4)เพื่อดูหมวดหมู่ที่สนใจ: ")) - 1

In [9]:
# Binary basis vectors สำหรับหมวดหมู่สินค้า
fashion_vector = np.array([1, 0, 1, 1, 0])
it_vector = np.array([0, 1, 0, 0, 1])

# เวกเตอร์ของ User
user1_vector = R[user_vector]

# คำนวณ dot product เพื่อหาคะแนนความสนใจแต่ละหมวด
fashion_score = np.dot(user1_vector, fashion_vector)
it_score = np.dot(user1_vector, it_vector)

print("User {} Vector:".format(user_vector + 1), user1_vector)
print("Fashion Vector:", fashion_vector)
print("IT Vector:", it_vector)

print("=" * 30)

print("คะแนนความสนใจหมวด Fashion ของ User {}: {:.0f}".format(user_vector + 1, fashion_score))
print("คะแนนความสนใจหมวด IT ของ User {}: {:.0f}".format(user_vector + 1, it_score))

if fashion_score > it_score:
    print("=" * 30)
    print("สรุป: User {} มีแนวโน้มสนใจหมวด Fashion มากกว่า".format(user_vector + 1))
elif it_score > fashion_score:
    print("=" * 30)
    print("สรุป: User {} มีแนวโน้มสนใจหมวด IT มากกว่า".format(user_vector + 1))
else:
    print("=" * 30)
    print("สรุป: User {} สนใจหมวด Fashion และ IT ใกล้เคียงกัน".format(user_vector + 1))

User 1 Vector: [5. 0. 4. 0. 1.]
Fashion Vector: [1 0 1 1 0]
IT Vector: [0 1 0 0 1]
คะแนนความสนใจหมวด Fashion ของ User 1: 9
คะแนนความสนใจหมวด IT ของ User 1: 1
สรุป: User 1 มีแนวโน้มสนใจหมวด Fashion มากกว่า


## สรุปโครงงาน

โครงงานนี้แสดงให้เห็นว่า Linear Algebra สามารถนำไปใช้สร้างระบบแนะนำสินค้าได้จริง โดยมีขั้นตอนสำคัญคือ

- แทนพฤติกรรมผู้ใช้และสินค้าเป็นเมทริกซ์ $R$

- ใช้ SVD เพื่อแยกเมทริกซ์เป็น $U \Sigma V^T$ และค้นหา latent features

- ใช้ Truncated SVD เพื่อลดมิติและสร้างเมทริกซ์ประมาณ $\hat{R}$

- แนะนำสินค้าจาก predicted score ที่สูงที่สุดสำหรับสินค้าที่ผู้ใช้ยังไม่เคยมีปฏิสัมพันธ์

- ใช้ Cosine Similarity เพื่อวัดความคล้ายกันของผู้ใช้ใน Inner Product Space

- ใช้ dot product และ Linear Combination เพื่อวิเคราะห์ความสนใจตามหมวดหมู่

ดังนั้น SVD ไม่ได้เป็นเพียงสูตรทางคณิตศาสตร์ แต่เป็นเครื่องมือสำคัญที่ช่วยให้ระบบแนะนำสินค้าเข้าใจรูปแบบความสนใจที่ซ่อนอยู่ในข้อมูลได้